# Padel Winner Classification\n\nPredict the winner between two pairs for a selected tournament and round using historical match statistics and head-to-head history.

In [ ]:
import re\nimport numpy as np\nimport pandas as pd\nimport pyodbc\n\nfrom collections import defaultdict\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import classification_report, accuracy_score\nfrom sklearn.model_selection import train_test_split\n

In [ ]:
SERVER = r'DESKTOP-QJ70MNR\\MSSQLSERVER1'\nDATABASE = 'DW_padel'\n\nconn_str = (\n    f'DRIVER={{SQL Server}};'\n    f'SERVER={SERVER};'\n    f'DATABASE={DATABASE};'\n    f'Trusted_Connection=yes;'\n)\n\nconn = pyodbc.connect(conn_str)\nprint('✅ Connected to SQL Server')

In [ ]:
query = '''\nSELECT\n    match_id,\n    match_number,\n    tournament_name,\n    round,\n    winner,\n    date,\n    team1_player1_name,\n    team1_player2_name,\n    team2_player1_name,\n    team2_player2_name,\n    aces_t1, aces_t2,\n    double_faults_t1, double_faults_t2,\n    won_on_1st_serve_t1, won_on_1st_serve_t2,\n    won_on_2nd_serve_t1, won_on_2nd_serve_t2,\n    total_points_won_t1, total_points_won_t2,\n    break_points_converted_t1, break_points_converted_t2,\n    total_won_on_serve_t1, total_won_on_serve_t2,\n    total_won_on_return_t1, total_won_on_return_t2\nFROM [DW_padel].[dbo].[matches_with_views_interactions]\nWHERE winner IN ('team_1', 'team_2')\n'''\n\ndf = pd.read_sql(query, conn)\ndf['date'] = pd.to_datetime(df['date'], errors='coerce')\ndf = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)\nprint(f'✅ Loaded {len(df)} matches')\ndf.head()

In [ ]:
def clean_tournament_name(name: str) -> str:\n    if pd.isna(name):\n        return ''\n    return re.sub(r'\\s+\\d{4}$', '', str(name)).strip()\n\ndef mk_pair(p1, p2):\n    p1 = str(p1).strip()\n    p2 = str(p2).strip()\n    return f'{p1} & {p2}'\n\nSTAT_BASES = [\n    'aces', 'double_faults', 'won_on_1st_serve', 'won_on_2nd_serve',\n    'total_points_won', 'break_points_converted', 'total_won_on_serve', 'total_won_on_return'\n]\n\ndf['tournament_clean'] = df['tournament_name'].apply(clean_tournament_name)\ndf['team1_pair'] = df.apply(lambda r: mk_pair(r['team1_player1_name'], r['team1_player2_name']), axis=1)\ndf['team2_pair'] = df.apply(lambda r: mk_pair(r['team2_player1_name'], r['team2_player2_name']), axis=1)

In [ ]:
pair_stats = defaultdict(lambda: {\n    'matches': 0,\n    'wins': 0,\n    'recent': [],\n    **{f'{s}_sum': 0.0 for s in STAT_BASES}\n})\nh2h = defaultdict(lambda: {'a_wins': 0, 'b_wins': 0, 'matches': 0})\n\nrows = []\n\nfor _, r in df.iterrows():\n    t1 = r['team1_pair']\n    t2 = r['team2_pair']\n    key = tuple(sorted([t1, t2]))\n\n    s1 = pair_stats[t1]\n    s2 = pair_stats[t2]\n    h = h2h[key]\n\n    feat = {\n        'tournament_clean': r['tournament_clean'],\n        'round': str(r['round']),\n        't1_prev_matches': s1['matches'],\n        't2_prev_matches': s2['matches'],\n        't1_prev_win_rate': s1['wins'] / s1['matches'] if s1['matches'] else 0.5,\n        't2_prev_win_rate': s2['wins'] / s2['matches'] if s2['matches'] else 0.5,\n        't1_recent_form': np.mean(s1['recent'][-5:]) if s1['recent'] else 0.5,\n        't2_recent_form': np.mean(s2['recent'][-5:]) if s2['recent'] else 0.5,\n    }\n\n    if key[0] == t1:\n        feat['t1_h2h_wins'] = h['a_wins']\n        feat['t2_h2h_wins'] = h['b_wins']\n    else:\n        feat['t1_h2h_wins'] = h['b_wins']\n        feat['t2_h2h_wins'] = h['a_wins']\n    feat['h2h_matches'] = h['matches']\n\n    for stat in STAT_BASES:\n        feat[f't1_avg_{stat}'] = s1[f'{stat}_sum'] / s1['matches'] if s1['matches'] else 0.0\n        feat[f't2_avg_{stat}'] = s2[f'{stat}_sum'] / s2['matches'] if s2['matches'] else 0.0\n\n    y = 1 if r['winner'] == 'team_1' else 0\n    rows.append((feat, y))\n\n    t1_win = 1 if y == 1 else 0\n    t2_win = 1 - t1_win\n\n    s1['matches'] += 1\n    s2['matches'] += 1\n    s1['wins'] += t1_win\n    s2['wins'] += t2_win\n    s1['recent'].append(t1_win)\n    s2['recent'].append(t2_win)\n\n    for stat in STAT_BASES:\n        s1[f'{stat}_sum'] += float(r[f'{stat}_t1']) if pd.notna(r[f'{stat}_t1']) else 0.0\n        s2[f'{stat}_sum'] += float(r[f'{stat}_t2']) if pd.notna(r[f'{stat}_t2']) else 0.0\n\n    if key[0] == t1:\n        h['a_wins'] += t1_win\n        h['b_wins'] += t2_win\n    else:\n        h['a_wins'] += t2_win\n        h['b_wins'] += t1_win\n    h['matches'] += 1\n\nX = pd.DataFrame([x for x, _ in rows])\ny = pd.Series([yy for _, yy in rows], name='target')\n\nX = pd.get_dummies(X, columns=['tournament_clean', 'round'], drop_first=False)\nprint(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)\n\nclf = RandomForestClassifier(n_estimators=350, max_depth=12, random_state=42, class_weight='balanced')\nclf.fit(X_train, y_train)\n\npred = clf.predict(X_test)\nprint('Accuracy:', round(accuracy_score(y_test, pred), 4))\nprint(classification_report(y_test, pred, target_names=['team_2_wins', 'team_1_wins']))

In [ ]:
def build_prediction_row(team1_pair, team2_pair, tournament_clean, round_name):\n    s1 = pair_stats[team1_pair]\n    s2 = pair_stats[team2_pair]\n    key = tuple(sorted([team1_pair, team2_pair]))\n    h = h2h[key]\n\n    row = {\n        't1_prev_matches': s1['matches'],\n        't2_prev_matches': s2['matches'],\n        't1_prev_win_rate': s1['wins'] / s1['matches'] if s1['matches'] else 0.5,\n        't2_prev_win_rate': s2['wins'] / s2['matches'] if s2['matches'] else 0.5,\n        't1_recent_form': np.mean(s1['recent'][-5:]) if s1['recent'] else 0.5,\n        't2_recent_form': np.mean(s2['recent'][-5:]) if s2['recent'] else 0.5,\n        'h2h_matches': h['matches'],\n    }\n\n    if key[0] == team1_pair:\n        row['t1_h2h_wins'] = h['a_wins']\n        row['t2_h2h_wins'] = h['b_wins']\n    else:\n        row['t1_h2h_wins'] = h['b_wins']\n        row['t2_h2h_wins'] = h['a_wins']\n\n    for stat in STAT_BASES:\n        row[f't1_avg_{stat}'] = s1[f'{stat}_sum'] / s1['matches'] if s1['matches'] else 0.0\n        row[f't2_avg_{stat}'] = s2[f'{stat}_sum'] / s2['matches'] if s2['matches'] else 0.0\n\n    row[f'tournament_clean_{tournament_clean}'] = 1\n    row[f'round_{round_name}'] = 1\n\n    pred_df = pd.DataFrame([row])\n    pred_df = pred_df.reindex(columns=X.columns, fill_value=0)\n    return pred_df\n\nteam_1 = 'Arturo Coello & Agustin Tapia'\nteam_2 = 'Alejandro Galan & Federico Chingotto'\nselected_tournament = 'Riyadh Season P1'\nselected_round = 'Finals'\n\nX_new = build_prediction_row(team_1, team_2, selected_tournament, selected_round)\np_team1 = clf.predict_proba(X_new)[0][1]\nprint(f'Team 1 win probability: {p_team1:.2%}')\nprint('Predicted winner:', team_1 if p_team1 >= 0.5 else team_2)